In [ ]:
# =========================
# Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================
# APOLLO INPUT PREPARATION
# Based on Apollo paper:
# 44.1 kHz, MONO, MP3 (96 kbps), then WAV
# ============================================

!apt-get -qq install -y ffmpeg
!pip install -q librosa soundfile numpy

import os
import glob
import tempfile
import subprocess
import numpy as np
import librosa
import soundfile as sf

# ============================================
# PATHS
# ============================================
dsp_input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/AudioSR_Output"
apollo_input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR"

os.makedirs(apollo_input_dir, exist_ok=True)

# ============================================
# SETTINGS
# ============================================
target_sr = 44100
mp3_bitrate = "96k"
peak_target = 0.95
hard_clip = 0.999

# ============================================
# FUNCTIONS
# ============================================
def to_mono(y):
    if y.ndim == 1:
        return y
    else:
        return np.mean(y, axis=0)

def peak_normalize(y, target_peak=0.95):
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y * (target_peak / peak)
    return y

# ============================================
# FIND FILES
# ============================================
wav_files = sorted(glob.glob(os.path.join(dsp_input_dir, "*.wav")))
print("Found files:", len(wav_files))

# ============================================
# PROCESS
# ============================================
for in_wav in wav_files:
    name = os.path.splitext(os.path.basename(in_wav))[0]
    out_wav = os.path.join(apollo_input_dir, name + "_apollo_input.wav")

    print("\nProcessing:", name)

    # -------------------------
    # 1) LOAD
    # -------------------------
    y, sr = librosa.load(in_wav, sr=None, mono=False)

    # -------------------------
    # 2) TO MONO
    # -------------------------
    y = to_mono(y)

    # -------------------------
    # 3) RESAMPLE TO 44.1 kHz
    # -------------------------
    if sr != target_sr:
        y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

    # -------------------------
    # 4) NORMALIZE
    # -------------------------
    y = peak_normalize(y, peak_target)
    y = np.clip(y, -hard_clip, hard_clip)

    # -------------------------
    # 5) MP3 ENCODE → DECODE
    # -------------------------
    with tempfile.TemporaryDirectory() as tmpdir:
        temp_wav = os.path.join(tmpdir, "temp.wav")
        temp_mp3 = os.path.join(tmpdir, "temp.mp3")

        sf.write(temp_wav, y, target_sr)

        # encode MP3
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", temp_wav,
            "-codec:a", "libmp3lame",
            "-b:a", mp3_bitrate,
            temp_mp3
        ], check=True)

        # decode back to WAV (Apollo input)
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-i", temp_mp3,
            "-ar", str(target_sr),
            "-ac", "1",
            "-sample_fmt", "s16",
            out_wav
        ], check=True)

    print("Saved:", out_wav)

print("\nAll files processed for Apollo input.")

Found files: 30

Processing: ambient_01
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/ambient_01_apollo_input.wav

Processing: ambient_02
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/ambient_02_apollo_input.wav

Processing: ambient_03
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/ambient_03_apollo_input.wav

Processing: ambient_04
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/ambient_04_apollo_input.wav

Processing: ambient_05
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/ambient_05_apollo_input.wav

Processing: drums_01
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/drums_01_apollo_input.wav

Processing: drums_02
Saved: /content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR/drums_02_apollo_input.wav

Processing: drums_03
Saved: /content/dr

In [ ]:
# =========================
# clone Apollo
# =========================
%cd /content
!rm -rf Apollo
!git clone https://github.com/JusperLee/Apollo.git
%cd /content/Apollo

!ls

!pip install -U pip
!pip install -e .
!pip install torch torchaudio librosa soundfile huggingface_hub omegaconf PyYAML scipy

import os
import glob
import torch
import torchaudio
import look2hear.models

# =========================
# Paths
# =========================
apollo_input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Degraded/Apollo_96_From_AudioSR"
apollo_output_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Apollo_Output_From_AudioSR_To_96k"
apollo_model_path = "/content/drive/MyDrive/MusicGen_Project/apollo_model/pytorch_model.bin"


os.makedirs(apollo_output_dir, exist_ok=True)

# =========================
# Load Apollo model
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = look2hear.models.BaseModel.from_pretrain(
    apollo_model_path,
    sr=44100,
    win=20,
    feature_dim=256,
    layer=6
).to(device)

model.eval()
print("Apollo loaded successfully")

# =========================
# Audio helpers
# =========================
def load_audio(path, device):
    audio, sr = torchaudio.load(path)
    if sr != 44100:
        raise ValueError(f"{os.path.basename(path)} has sample rate {sr}, expected 44100")
    return audio.unsqueeze(0).to(device)   # [1, channels, samples]

def save_audio(path, audio, samplerate=44100):
    audio = audio.squeeze(0).detach().cpu()   # [channels, samples]
    torchaudio.save(path, audio, samplerate)

# =========================
# Run Apollo on all files
# =========================
wav_files = sorted(glob.glob(os.path.join(apollo_input_dir, "*.wav")))
print("Found files:", len(wav_files))

failed = []

for in_wav in wav_files:
    base = os.path.splitext(os.path.basename(in_wav))[0]
    out_name = base.replace("_apollo_input", "_apollo_output") + ".wav"
    out_wav = os.path.join(apollo_output_dir, out_name)

    print(f"\nProcessing: {os.path.basename(in_wav)}")

    try:
        test_data = load_audio(in_wav, device)

        with torch.no_grad():
            restored = model(test_data)

        save_audio(out_wav, restored, samplerate=44100)
        print("Saved:", out_wav)

    except Exception as e:
        print("FAILED:", os.path.basename(in_wav), "->", e)
        failed.append((in_wav, str(e)))

# =========================
# 8) Summary
# =========================
print("\nAll files processed.")
print("Success:", len(wav_files) - len(failed))
print("Failed:", len(failed))

if failed:
    print("\nFailed files:")
    for f, err in failed:
        print("-", os.path.basename(f), ":", err)

/content
Cloning into 'Apollo'...
remote: Enumerating objects: 232, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 232 (delta 20), reused 15 (delta 13), pack-reused 196 (from 1)
Receiving objects: 100% (232/232), 88.91 MiB | 28.36 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/Apollo
asserts  index.html    LICENSE	  look2hear.yml  test.py
configs  inference.py  look2hear  README.md	 train.py
Obtaining file:///content/Apollo
ERROR: file:///content/Apollo does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Device: cuda
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, np.int64(47)] 80
Apollo loaded successfully
Found files: 30

Processing: ambient_01_apollo_input.wav
Saved: /content/drive/

In [ ]:
# =========================
# Listen BEFORE vs AFTER for all files
# =========================
import os
import glob
from IPython.display import Audio, display

input_dir = "/content/drive/MyDrive/MusicGen_Project/Data/AudioSR_Output"
output_dir = "/content/drive/MyDrive/MusicGen_Project/Data/Apollo_Output_From_AudioSR_To_96k"

input_files = sorted(glob.glob(os.path.join(input_dir, "*.wav")))

print("Found files:", len(input_files))

for in_file in input_files:
    base = os.path.basename(in_file)
    stem = os.path.splitext(base)[0]
    out_file = os.path.join(output_dir, f"{stem}_apollo_output.wav")

    print("\n==============================")
    print("File:", base)
    print("==============================")

    if os.path.exists(out_file):
        print("INPUT (AudioSR output)")
        display(Audio(in_file))

        print("APOLLO OUTPUT")
        display(Audio(out_file))
    else:
        print("Output file not found for:", base)
        print("Expected file:", out_file)

Output hidden; open in https://colab.research.google.com to view.